In [12]:
# Import library

import glob, json, pickle
import numpy as np
import pandas as pd
import mat73
from statsmodels.stats.multitest import fdrcorrection
from tqdm import tqdm
import scipy.stats as stats

In [2]:
# Original sample
x = np.array([5.1, 5.3, 5.8, 6.0, 5.6])
n_boot = 10000  # Number of bootstrap samples
boot_means = []

# Bootstrap resampling
for _ in range(n_boot):
    sample = np.random.choice(x, size=len(x), replace=True)
    boot_means.append(np.mean(sample))

# Compute 95% CI from percentiles
ci_lower = np.percentile(boot_means, 2.5)
ci_upper = np.percentile(boot_means, 97.5)

print(np.mean(boot_means))
print(f"95% Bootstrap CI for mean: ({ci_lower:.3f}, {ci_upper:.3f})")

5.56005
95% Bootstrap CI for mean: (5.280, 5.840)


In [10]:
# Load data

path = "/Users/woojaejeong/Desktop/Data/USC/DARPA-NEAT/Code/Decoding/svm/Result/svmLatencyBoots_commonPCA.mat"

Result = mat73.loadmat(path)

metrics = {
    "Lat": {"con": Result["conLat"], "dep": Result["depLat"], "sui": Result["suiLat"]},
    "PeakLat": {
        "con": Result["conPeakLat"],
        "dep": Result["depPeakLat"],
        "sui": Result["suiPeakLat"],
    },
    "Amp": {
        "con": Result["conPeak"],
        "dep": Result["depPeak"],
        "sui": Result["suiPeak"],
    },
    "Off": {"con": Result["conOff"], "dep": Result["depOff"], "sui": Result["suiOff"]},
}

group_pairs = [("con", "dep"), ("con", "sui"), ("dep", "sui")]
n_perm = 1000

# Track results
p_values = []
comparison_labels = []

# Loop over metrics and group pairs
for metric_name, group_data in metrics.items():
    for g1, g2 in group_pairs:
        x = group_data[g1]
        y = group_data[g2]

        obs_diff = np.mean(x) - np.mean(y)
        combined = np.concatenate([x, y])
        n_x = len(x)

        perm_diffs = []
        for _ in range(n_perm - 1):
            perm = np.random.permutation(combined)
            perm_diff = np.mean(perm[:n_x]) - np.mean(perm[n_x:])
            perm_diffs.append(perm_diff)

        perm_diffs = np.array(perm_diffs)
        p = (np.sum(np.abs(perm_diffs) >= np.abs(obs_diff)) + 1) / n_perm
        p_values.append(p)
        comparison_labels.append((metric_name, g1, g2))

# FDR correction across all metrics and comparisons
p_values = np.array(p_values)
rejected, pvals_corrected = fdrcorrection(p_values, alpha=0.01)

# Reporting
print("=== FDR-Corrected Results ===")
for i, (metric, g1, g2) in enumerate(comparison_labels):
    print(
        f"{metric}: {g1} vs {g2} → raw p = {p_values[i]:.4f}, FDR p = {pvals_corrected[i]:.4f}, significant: {rejected[i]}"
    )

=== FDR-Corrected Results ===
Lat: con vs dep → raw p = 0.0010, FDR p = 0.0015, significant: True
Lat: con vs sui → raw p = 0.0010, FDR p = 0.0015, significant: True
Lat: dep vs sui → raw p = 0.0040, FDR p = 0.0053, significant: True
PeakLat: con vs dep → raw p = 0.1200, FDR p = 0.1309, significant: False
PeakLat: con vs sui → raw p = 0.6130, FDR p = 0.6130, significant: False
PeakLat: dep vs sui → raw p = 0.0170, FDR p = 0.0204, significant: False
Amp: con vs dep → raw p = 0.0010, FDR p = 0.0015, significant: True
Amp: con vs sui → raw p = 0.0010, FDR p = 0.0015, significant: True
Amp: dep vs sui → raw p = 0.0010, FDR p = 0.0015, significant: True
Off: con vs dep → raw p = 0.0010, FDR p = 0.0015, significant: True
Off: con vs sui → raw p = 0.0010, FDR p = 0.0015, significant: True
Off: dep vs sui → raw p = 0.0010, FDR p = 0.0015, significant: True


In [ ]:
# ============================================================
# Load data
# ============================================================
path = "/Users/woojaejeong/Desktop/Data/USC/DARPA-NEAT/Code/Decoding/svm/Result/svmLatencyBoots_commonPCA.mat"
Result = mat73.loadmat(path)

metrics = {
    "Lat": {"con": Result["conLat"], "dep": Result["depLat"], "sui": Result["suiLat"]},
    "PeakLat": {
        "con": Result["conPeakLat"],
        "dep": Result["depPeakLat"],
        "sui": Result["suiPeakLat"],
    },
    "Amp": {
        "con": Result["conPeak"],
        "dep": Result["depPeak"],
        "sui": Result["suiPeak"],
    },
    "Off": {"con": Result["conOff"], "dep": Result["depOff"], "sui": Result["suiOff"]},
}

group_pairs = [("con", "dep"), ("con", "sui"), ("dep", "sui")]


# ============================================================
# Cliff's delta for bootstrap distributions (paired draws)
#   delta = P(X>Y) - P(X<Y) = 2*P(X>Y) + P(X==Y) - 1
# ============================================================
def _to_1d(a):
    return np.asarray(a).ravel()


def pct_ci(x, alpha=0.05):
    lo = np.nanpercentile(x, 100 * (alpha / 2))
    hi = np.nanpercentile(x, 100 * (1 - alpha / 2))
    return float(lo), float(hi)


def cliffs_delta_paired_boot(x_draws, y_draws, alpha=0.05):
    """
    Computes Cliff's delta using paired bootstrap draws:
      delta = P(x>y) - P(x<y)
    Also returns a percentile CI by computing delta on each draw (delta_i in {-1,0,1}).
    """
    x = _to_1d(x_draws)
    y = _to_1d(y_draws)

    n = min(len(x), len(y))
    x = x[:n]
    y = y[:n]

    gt = x > y
    lt = x < y

    delta = float(np.nanmean(gt.astype(float) - lt.astype(float)))

    # draw-wise contributions: +1 if gt, -1 if lt, 0 if tie/NaN-comparison excluded by nanmean
    contrib = gt.astype(float) - lt.astype(float)
    ci_lo, ci_hi = pct_ci(contrib, alpha=alpha)

    # Probability of superiority (optional to keep for sanity checks)
    pos = float(np.nanmean(gt))

    return {
        "n_boot": n,
        "cliffs_delta": delta,
        "ci_lo": ci_lo,
        "ci_hi": ci_hi,
        "PoS": pos,
    }


# ============================================================
# Compute for all metrics and pairs
# ============================================================
rows = []
alpha = 0.05  # 95% CI

for metric_name, group_data in metrics.items():
    for g1, g2 in group_pairs:
        res = cliffs_delta_paired_boot(group_data[g1], group_data[g2], alpha=alpha)
        res.update({"metric": metric_name, "pair": f"{g1} vs {g2}"})
        rows.append(res)

df = pd.DataFrame(
    rows, columns=["metric", "pair", "n_boot", "cliffs_delta", "ci_lo", "ci_hi", "PoS"]
)

pd.set_option("display.max_rows", None)
pd.set_option("display.width", 140)
pd.set_option("display.precision", 4)

print(df.to_string(index=False))

# Optional: save CSV
# out_csv = "/Users/woojaejeong/Desktop/cliffs_delta_bootstrap.csv"
# df.to_csv(out_csv, index=False)
# print(f"\nSaved: {out_csv}")

 metric       pair  n_boot  cliffs_delta  ci_lo  ci_hi   PoS
    Lat con vs dep    1000         0.735   -1.0    1.0 0.861
    Lat con vs sui    1000         0.832   -1.0    1.0 0.911
    Lat dep vs sui    1000        -0.062   -1.0    1.0 0.434
PeakLat con vs dep    1000         0.308   -1.0    1.0 0.644
PeakLat con vs sui    1000        -0.055   -1.0    1.0 0.439
PeakLat dep vs sui    1000        -0.475   -1.0    1.0 0.258
    Amp con vs dep    1000        -0.984   -1.0   -1.0 0.008
    Amp con vs sui    1000        -1.000   -1.0   -1.0 0.000
    Amp dep vs sui    1000        -0.886   -1.0    1.0 0.057
    Off con vs dep    1000        -0.824   -1.0    1.0 0.082
    Off con vs sui    1000        -0.919   -1.0    1.0 0.035
    Off dep vs sui    1000        -0.583   -1.0    1.0 0.200


In [ ]:
p_values = np.array([0.119, 0.611, 0.031])
_, pvals_corrected = fdrcorrection(p_values, alpha=0.05)
print(pvals_corrected)

[0.1785 0.611  0.093 ]


In [160]:
dat = metrics["Off"]["sui"]

ci_lower = np.percentile(dat, 2.5)
ci_upper = np.percentile(dat, 97.5)

print(f"Mean of null: {np.mean(dat)}")
print(f"95% CI of null: [{ci_lower:.3f}, {ci_upper:.3f}]")

Mean of null: 736.372
95% CI of null: [609.000, 861.000]


In [11]:
# Load data

path = "/Users/woojaejeong/Desktop/Data/USC/DARPA-NEAT/Code/Decoding/svm/Result/svmLatencyBoots_commonPCA.mat"

Result = mat73.loadmat(path)

metrics = {
    "Lat": {
        "con": Result["conOff"] - Result["conLat"],
        "dep": Result["depOff"] - Result["depLat"],
        "sui": Result["suiOff"] - Result["suiLat"],
    },
}

group_pairs = [("con", "dep"), ("con", "sui"), ("dep", "sui")]
n_perm = 1000

# Track results
p_values = []
comparison_labels = []

# Loop over metrics and group pairs
for metric_name, group_data in metrics.items():
    for g1, g2 in group_pairs:
        x = group_data[g1]
        y = group_data[g2]

        obs_diff = np.mean(x) - np.mean(y)
        combined = np.concatenate([x, y])
        n_x = len(x)

        perm_diffs = []
        for _ in range(n_perm - 1):
            perm = np.random.permutation(combined)
            perm_diff = np.mean(perm[:n_x]) - np.mean(perm[n_x:])
            perm_diffs.append(perm_diff)

        perm_diffs = np.array(perm_diffs)
        p = (np.sum(np.abs(perm_diffs) >= np.abs(obs_diff)) + 1) / n_perm
        p_values.append(p)
        comparison_labels.append((metric_name, g1, g2))

# FDR correction across all metrics and comparisons
p_values = np.array(p_values)
rejected, pvals_corrected = fdrcorrection(p_values, alpha=0.01)

# Reporting
print("=== FDR-Corrected Results ===")
for i, (metric, g1, g2) in enumerate(comparison_labels):
    print(
        f"{metric}: {g1} vs {g2} → raw p = {p_values[i]:.4f}, FDR p = {pvals_corrected[i]:.4f}, significant: {rejected[i]}"
    )

=== FDR-Corrected Results ===
Lat: con vs dep → raw p = 0.0010, FDR p = 0.0010, significant: True
Lat: con vs sui → raw p = 0.0010, FDR p = 0.0010, significant: True
Lat: dep vs sui → raw p = 0.0010, FDR p = 0.0010, significant: True


In [ ]:
# ============================================================
# Load data
# ============================================================
path = "/Users/woojaejeong/Desktop/Data/USC/DARPA-NEAT/Code/Decoding/svm/Result/svmLatencyBoots_commonPCA.mat"
Result = mat73.loadmat(path)

# Duration metric: Offset − Onset (bootstrap draws)
metrics = {
    "Duration": {
        "con": Result["conOff"] - Result["conLat"],
        "dep": Result["depOff"] - Result["depLat"],
        "sui": Result["suiOff"] - Result["suiLat"],
    }
}

group_pairs = [("con", "dep"), ("con", "sui"), ("dep", "sui")]


# ============================================================
# Cliff's delta (paired bootstrap version)
# ============================================================
def _to_1d(a):
    return np.asarray(a).ravel()


def pct_ci(x, alpha=0.05):
    lo = np.nanpercentile(x, 100 * (alpha / 2))
    hi = np.nanpercentile(x, 100 * (1 - alpha / 2))
    return float(lo), float(hi)


def cliffs_delta_paired_boot(x_draws, y_draws, alpha=0.05):
    """
    Cliff's delta computed on paired bootstrap draws:
      delta = P(x > y) - P(x < y)
    CI obtained from percentile CI of per-draw contributions.
    """
    x = _to_1d(x_draws)
    y = _to_1d(y_draws)

    n = min(len(x), len(y))
    x = x[:n]
    y = y[:n]

    gt = x > y
    lt = x < y

    delta = float(np.nanmean(gt.astype(float) - lt.astype(float)))

    contrib = gt.astype(float) - lt.astype(float)
    ci_lo, ci_hi = pct_ci(contrib, alpha=alpha)

    return {
        "n_boot": n,
        "cliffs_delta": delta,
        "ci_lo": ci_lo,
        "ci_hi": ci_hi,
    }


# ============================================================
# Compute Cliff's delta for all pairs
# ============================================================
rows = []
alpha = 0.05  # 95% CI

for metric_name, group_data in metrics.items():
    for g1, g2 in group_pairs:
        res = cliffs_delta_paired_boot(group_data[g1], group_data[g2], alpha=alpha)
        res.update({"metric": metric_name, "pair": f"{g1} vs {g2}"})
        rows.append(res)

df = pd.DataFrame(
    rows, columns=["metric", "pair", "n_boot", "cliffs_delta", "ci_lo", "ci_hi"]
)

pd.set_option("display.max_rows", None)
pd.set_option("display.width", 120)
pd.set_option("display.precision", 4)

print(df.to_string(index=False))

# Optional: save CSV
# out_csv = "/Users/woojaejeong/Desktop/cliffs_delta_duration.csv"
# df.to_csv(out_csv, index=False)
# print(f"\nSaved: {out_csv}")

  metric       pair  n_boot  cliffs_delta  ci_lo  ci_hi
Duration con vs dep    1000        -0.876   -1.0    1.0
Duration con vs sui    1000        -0.961   -1.0   -1.0
Duration dep vs sui    1000        -0.567   -1.0    1.0


In [150]:
dat = metrics["Lat"]["sui"]

ci_lower = np.percentile(dat, 2.5)
ci_upper = np.percentile(dat, 97.5)

print(f"Mean of null: {np.mean(dat)}")
print(f"95% CI of null: [{ci_lower:.3f}, {ci_upper:.3f}]")

Mean of null: 414.116
95% CI of null: [263.900, 548.100]


In [154]:
# Load data correlation within class

path = "/Users/woojaejeong/Desktop/Data/USC/DARPA-NEAT/Code/Decoding/svm/Result/group_corr.mat"

Result = mat73.loadmat(path)

metrics = {
    "con": {"pc1": Result["rho_pc1_con"], "pc2": Result["rho_pc2_con"]},
    "dep": {"pc1": Result["rho_pc1_dep"], "pc2": Result["rho_pc2_dep"]},
    "sui": {"pc1": Result["rho_pc1_sui"], "pc2": Result["rho_pc2_sui"]},
}

group_pairs = [("pc1", "pc2")]
n_perm = 1000

# Track results
p_values = []
comparison_labels = []

# Loop over metrics and group pairs
for metric_name, group_data in metrics.items():
    for g1, g2 in group_pairs:
        x = group_data[g1]
        y = group_data[g2]

        obs_diff = np.mean(x) - np.mean(y)
        combined = np.concatenate([x, y])
        n_x = len(x)

        perm_diffs = []
        for _ in range(n_perm - 1):
            perm = np.random.permutation(combined)
            perm_diff = np.mean(perm[:n_x]) - np.mean(perm[n_x:])
            perm_diffs.append(perm_diff)

        perm_diffs = np.array(perm_diffs)
        p = (np.sum(np.abs(perm_diffs) >= np.abs(obs_diff)) + 1) / n_perm
        p_values.append(p)
        comparison_labels.append((metric_name, g1, g2))

# FDR correction across all metrics and comparisons
p_values = np.array(p_values)
rejected, pvals_corrected = fdrcorrection(p_values, alpha=0.01)

# Reporting
print("=== FDR-Corrected Results ===")
for i, (metric, g1, g2) in enumerate(comparison_labels):
    print(
        f"{metric}: {g1} vs {g2} → raw p = {p_values[i]:.4f}, FDR p = {pvals_corrected[i]:.4f}, significant: {rejected[i]}"
    )

=== FDR-Corrected Results ===
con: pc1 vs pc2 → raw p = 0.0010, FDR p = 0.0010, significant: True
dep: pc1 vs pc2 → raw p = 0.0010, FDR p = 0.0010, significant: True
sui: pc1 vs pc2 → raw p = 0.0010, FDR p = 0.0010, significant: True


In [ ]:
# ============================================================
# Load data correlation within class
# ============================================================
path = "/Users/woojaejeong/Desktop/Data/USC/DARPA-NEAT/Code/Decoding/svm/Result/group_corr.mat"
Result = mat73.loadmat(path)

metrics = {
    "con": {"pc1": Result["rho_pc1_con"], "pc2": Result["rho_pc2_con"]},
    "dep": {"pc1": Result["rho_pc1_dep"], "pc2": Result["rho_pc2_dep"]},
    "sui": {"pc1": Result["rho_pc1_sui"], "pc2": Result["rho_pc2_sui"]},
}

group_pairs = [("pc1", "pc2")]


# ============================================================
# Cliff's delta (unpaired, all-vs-all) for pc1 vs pc2 within each class
#   delta = P(PC1 > PC2) - P(PC1 < PC2)
# ============================================================
def _to_1d(a):
    return np.asarray(a).ravel()


def cliffs_delta_unpaired(x, y):
    x = _to_1d(x)
    y = _to_1d(y)

    # remove NaNs/infs (important for correlations)
    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    if len(x) == 0 or len(y) == 0:
        return np.nan, np.nan  # (delta, PoS)

    gt = np.mean(x[:, None] > y[None, :])
    lt = np.mean(x[:, None] < y[None, :])

    delta = float(gt - lt)
    pos = float(gt)  # P(PC1 > PC2)
    return delta, pos


# ============================================================
# Compute Cliff's delta for each class
# ============================================================
rows = []
for cls_name, pcs in metrics.items():
    x = pcs["pc1"]
    y = pcs["pc2"]
    delta, pos = cliffs_delta_unpaired(x, y)
    rows.append(
        {
            "class": cls_name,
            "comparison": "pc1 vs pc2",
            "cliffs_delta": delta,
            "PoS": pos,
        }
    )

df = pd.DataFrame(rows, columns=["class", "comparison", "cliffs_delta", "PoS"])

pd.set_option("display.max_rows", None)
pd.set_option("display.width", 90)
pd.set_option("display.precision", 4)

print(df.to_string(index=False))

# Optional: save CSV
# out_csv = "/Users/woojaejeong/Desktop/cliffs_delta_within_class_pc1_pc2.csv"
# df.to_csv(out_csv, index=False)
# print(f"\nSaved: {out_csv}")

class comparison  cliffs_delta    PoS
  con pc1 vs pc2       -0.7922 0.1039
  dep pc1 vs pc2       -0.4562 0.2719
  sui pc1 vs pc2       -0.7707 0.1147


In [172]:
# Load data correlation

path = "/Users/woojaejeong/Desktop/Data/USC/DARPA-NEAT/Code/Decoding/svm/Result/group_corr.mat"

Result = mat73.loadmat(path)

metrics = {
    "pc1": {
        "con": Result["rho_pc1_con"],
        "dep": Result["rho_pc1_dep"],
        "sui": Result["rho_pc1_sui"],
    },
    "pc2": {
        "con": Result["rho_pc2_con"],
        "dep": Result["rho_pc2_dep"],
        "sui": Result["rho_pc2_sui"],
    },
}

group_pairs = [("con", "dep"), ("con", "sui"), ("dep", "sui")]
n_perm = 1000

# Track results
p_values = []
comparison_labels = []

# Loop over metrics and group pairs
for metric_name, group_data in metrics.items():
    for g1, g2 in group_pairs:
        x = group_data[g1]
        y = group_data[g2]

        obs_diff = np.mean(x) - np.mean(y)
        combined = np.concatenate([x, y])
        n_x = len(x)

        perm_diffs = []
        for _ in range(n_perm - 1):
            perm = np.random.permutation(combined)
            perm_diff = np.mean(perm[:n_x]) - np.mean(perm[n_x:])
            perm_diffs.append(perm_diff)

        perm_diffs = np.array(perm_diffs)
        p = (np.sum(np.abs(perm_diffs) >= np.abs(obs_diff)) + 1) / n_perm
        p_values.append(p)
        comparison_labels.append((metric_name, g1, g2))

# FDR correction across all metrics and comparisons
p_values = np.array(p_values)
rejected, pvals_corrected = fdrcorrection(p_values, alpha=0.01)

# Reporting
print("=== FDR-Corrected Results ===")
for i, (metric, g1, g2) in enumerate(comparison_labels):
    print(
        f"{metric}: {g1} vs {g2} → raw p = {p_values[i]:.4f}, FDR p = {pvals_corrected[i]:.4f}, significant: {rejected[i]}"
    )

=== FDR-Corrected Results ===
pc1: con vs dep → raw p = 0.0010, FDR p = 0.0010, significant: True
pc1: con vs sui → raw p = 0.0010, FDR p = 0.0010, significant: True
pc1: dep vs sui → raw p = 0.0010, FDR p = 0.0010, significant: True
pc2: con vs dep → raw p = 0.0010, FDR p = 0.0010, significant: True
pc2: con vs sui → raw p = 0.0010, FDR p = 0.0010, significant: True
pc2: dep vs sui → raw p = 0.0010, FDR p = 0.0010, significant: True


In [ ]:
# ============================================================
# Load data correlation
# ============================================================
path = "/Users/woojaejeong/Desktop/Data/USC/DARPA-NEAT/Code/Decoding/svm/Result/group_corr.mat"
Result = mat73.loadmat(path)

metrics = {
    "pc1": {
        "con": Result["rho_pc1_con"],
        "dep": Result["rho_pc1_dep"],
        "sui": Result["rho_pc1_sui"],
    },
    "pc2": {
        "con": Result["rho_pc2_con"],
        "dep": Result["rho_pc2_dep"],
        "sui": Result["rho_pc2_sui"],
    },
}

group_pairs = [("con", "dep"), ("con", "sui"), ("dep", "sui")]


# ============================================================
# Cliff's delta (standard unpaired, all-vs-all)
#   delta = P(X>Y) - P(X<Y)
# ============================================================
def _to_1d(a):
    return np.asarray(a).ravel()


def cliffs_delta_unpaired(x, y):
    """
    Unpaired Cliff's delta for two independent samples.
    Computes all pairwise comparisons (O(n*m)).
    """
    x = _to_1d(x)
    y = _to_1d(y)

    # remove NaNs (important for correlations)
    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    if len(x) == 0 or len(y) == 0:
        return np.nan, np.nan  # (delta, PoS)

    # Broadcast compare: n x m boolean matrices
    gt = (x[:, None] > y[None, :]).mean()
    lt = (x[:, None] < y[None, :]).mean()

    delta = float(gt - lt)
    pos = float(gt)  # Probability of superiority P(X>Y)
    return delta, pos


# ============================================================
# Compute Cliff's delta for all metrics and pairs
# ============================================================
rows = []
for metric_name, group_data in metrics.items():
    for g1, g2 in group_pairs:
        delta, pos = cliffs_delta_unpaired(group_data[g1], group_data[g2])
        rows.append(
            {
                "metric": metric_name,
                "pair": f"{g1} vs {g2}",
                "cliffs_delta": delta,
                "PoS": pos,
            }
        )

df = pd.DataFrame(rows, columns=["metric", "pair", "cliffs_delta", "PoS"])

pd.set_option("display.max_rows", None)
pd.set_option("display.width", 120)
pd.set_option("display.precision", 4)

print(df.to_string(index=False))

# Optional: save CSV
# out_csv = "/Users/woojaejeong/Desktop/cliffs_delta_corr.csv"
# df.to_csv(out_csv, index=False)
# print(f"\nSaved: {out_csv}")

metric       pair  cliffs_delta    PoS
   pc1 con vs dep       -0.9324 0.0338
   pc1 con vs sui       -0.9861 0.0070
   pc1 dep vs sui       -0.4179 0.2910
   pc2 con vs dep       -0.8250 0.0875
   pc2 con vs sui       -0.9824 0.0088
   pc2 dep vs sui       -0.7590 0.1205


In [176]:
dat = metrics["pc1"]["sui"]
dat2 = metrics["pc2"]["sui"]

ci_lower = np.percentile(dat, 2.5)
ci_upper = np.percentile(dat, 97.5)

print(f"Mean of null: {np.mean(dat)}")
print(f"95% CI of null: [{ci_lower:.3f}, {ci_upper:.3f}]")

ci_lower = np.percentile(dat2, 2.5)
ci_upper = np.percentile(dat2, 97.5)

print(f"Mean of null: {np.mean(dat2)}")
print(f"95% CI of null: [{ci_lower:.3f}, {ci_upper:.3f}]")

Mean of null: 0.6940969298187308
95% CI of null: [0.507, 0.817]
Mean of null: 0.8055810499968302
95% CI of null: [0.669, 0.896]
